In [ ]:
# Notebook 05: Multi-Agent Orchestration
#
!pip install langchain langchain-community langgraph -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!pip install langchain langchain-community chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.9 MB/s eta

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import Tool
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from pathlib import Path
import json

CHROMA_DIR = BASE_DIR / "data/vector_store"

print("Multi-agent setup initialized")

Multi-agent setup initialized


In [ ]:

# Load vector store
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="arxiv_papers"
)

print("Vector store loaded")

Vector store loaded


In [ ]:

# Retrieval Agent Tool
def retrieval_tool(query: str) -> str:
    """Search research papers for relevant information"""
    docs = vectorstore.similarity_search(query, k=5)

    results = []
    for i, doc in enumerate(docs, 1):
        results.append(f"{i}. {doc.metadata['title']}\n{doc.page_content[:200]}...\n")

    return "\n".join(results)

retrieval_agent_tool = Tool(
    name="PaperSearch",
    func=retrieval_tool,
    description="Search academic papers for information about a topic"
)


In [ ]:

# Citation Agent Tool
def citation_tool(arxiv_ids: str) -> str:
    """Format citations for papers"""
    ids = arxiv_ids.split(',')
    citations = []

    for arxiv_id in ids:
        # Query metadata
        results = vectorstore.get(
            where={"arxiv_id": arxiv_id.strip()}
        )
        if results and results['metadatas']:
            meta = results['metadatas'][0]
            citation = f"[{meta['arxiv_id']}] {meta['title']} - {meta['authors']}"
            citations.append(citation)

    return "\n".join(citations)

citation_agent_tool = Tool(
    name="FormatCitations",
    func=citation_tool,
    description="Format academic citations for papers given arxiv IDs"
)


In [ ]:

# Test tools
print("Testing retrieval tool...")
result = retrieval_tool("transformers in NLP")
print(result[:300])

print("\n" + "="*60)
print("Multi-agent orchestration ready!")


Testing retrieval tool...
1. Why Are Positional Encodings Nonessential for Deep Autoregressive Transformers? Revisiting a Petroglyph
Preprint arXiv:2402.03902. Zihang Dai, Zhilin Yang, Yiming Yang, William W Cohen, Jaime Carbonell, Quoc V Le, and Ruslan Salakhutdinov. 2019. Transformer-XL: Attentive lan- guage models beyond 

Multi-agent orchestration ready!
